# Vicon vs exo sensor kinematics — knee exoskeleton

Paper section *Influence of Kinematic Input Quality* (knee).

Canonical results live in `analysis/paper_outputs/knee_exo/` (exported by `compare_processed_knee_exo_id.ipynb` Section 3 paper tables). This notebook loads those tables and writes the kinematic-input-quality summary for the paper.

Per trial, compare:

1. **Encoder replay (sensor path)** — logged encoder angle + raw velocity, no input LPF, offline through `runs/0707_knee_finetune_balanced_lg_ra_rd/best_model.pt` vs GT (encoder–IK xcorr ID shift, 10 s trim).
2. **Vicon IK oracle** — `{subject_dir}/knee-exo/ik/{COND}_{speed}_ik.mot` knee angle + B-spline velocity through the **same** checkpoint vs the **same GT**.

Reference tables:
- `table1_summary.csv` — mean ± std across trials
- `table2_per_trial.csv` — per-trial encoder vs Vicon metrics
- `table3_overall.csv` — overall encoder vs Vicon summary

Regenerate the reference by re-running `compare_processed_knee_exo_id.ipynb` (Section 3 replay + paper export).


In [2]:
import re
from pathlib import Path

import numpy as np
import pandas as pd

PROJECT_ROOT = Path('/home/metamobility3/Jinwoo/os_kinetics').resolve()
REF_DIR = PROJECT_ROOT / 'analysis' / 'paper_outputs' / 'knee_exo'
OUT_DIR = PROJECT_ROOT / 'analysis' / 'paper_outputs' / 'kinematic_input_quality'
METRICS_CSV = PROJECT_ROOT / 'analysis' / 'cache' / 'vicon_vs_knee_exo_metrics.csv'
REPLAY_METRICS_CSV = PROJECT_ROOT / 'analysis' / 'cache' / 'compare_processed_knee_exo_id_replay_metrics.csv'

TABLE1 = REF_DIR / 'table1_summary.csv'
TABLE2 = REF_DIR / 'table2_per_trial.csv'
TABLE3 = REF_DIR / 'table3_overall.csv'

SUBJECT_TOKEN_TO_DIR = {
    'ab01_jinwoo': 'AB01_Jinwoo', 'ab02_oscar': 'AB02_Oscar', 'ab03_ilseung': 'AB03_Ilseung',
    'ab04_changseob': 'AB04_Changseob', 'ab05_maria': 'AB05_Maria', 'ab06_jimin': 'AB06_Jimin',
    'ab07_amy': 'AB07_Amy', 'ab08_seokhyun': 'AB08_Seokhyun',
}


def _parse_mean_std(s: str) -> tuple[float, float]:
    mean_s, std_s = str(s).split('±')
    return float(mean_s.strip()), float(std_s.strip())


def load_reference_tables() -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    for p in (TABLE1, TABLE2, TABLE3):
        if not p.is_file():
            raise FileNotFoundError(
                f'Missing {p}. Run compare_processed_knee_exo_id.ipynb paper export first.'
            )
    return pd.read_csv(TABLE1), pd.read_csv(TABLE2), pd.read_csv(TABLE3)


def attach_trial_stems(per_trial: pd.DataFrame, replay: pd.DataFrame) -> pd.DataFrame:
    replay = replay.copy()
    replay['trial_key'] = replay['subject'].astype(str) + '::' + replay['condition'].astype(str)
    per_trial = per_trial.copy()
    per_trial['trial_key'] = per_trial['subject'].astype(str) + '::' + per_trial['condition'].astype(str)
    merged = per_trial.merge(
        replay[['trial_key', 'trial', 'model_in_ang_rmse_deg', 'model_in_vel_rmse_deg_s']],
        on='trial_key', how='left', validate='one_to_one',
    )
    if merged['trial'].isna().any():
        missing = merged.loc[merged['trial'].isna(), 'trial_key'].tolist()
        raise KeyError(f'No replay trial stem for: {missing}')
    return merged


def build_metrics_df(per_trial: pd.DataFrame) -> pd.DataFrame:
    df = per_trial[~per_trial['encoder_replay_excluded'].astype(bool)].copy()
    df = df.rename(columns={
        'rmse_encoder_replay_nmpkg': 'rmse_sensor_nmpkg',
        'r2_encoder_replay_nmpkg': 'r2_sensor_nmpkg',
        'rmse_vicon_oracle_nmpkg': 'rmse_vicon_nmpkg',
        'r2_vicon_oracle_nmpkg': 'r2_vicon_nmpkg',
        'model_in_ang_rmse_deg': 'ang_rmse_deg_sensor_vicon',
    })
    return df


def summarize_pair(df: pd.DataFrame, label: str = 'overall') -> dict:
    s = df['rmse_sensor_nmpkg'].mean()
    r2s = df['r2_sensor_nmpkg'].mean()
    v = df['rmse_vicon_nmpkg'].mean()
    r2v = df['r2_vicon_nmpkg'].mean()
    return {
        'group': label,
        'n_trials': len(df),
        'rmse_sensor_nmpkg': float(s),
        'r2_sensor': float(r2s),
        'rmse_vicon_nmpkg': float(v),
        'r2_vicon': float(r2v),
        'rmse_reduction_pct': float((s - v) / s * 100.0),
        'r2_increase': float(r2v - r2s),
    }


def write_paper_outputs(df: pd.DataFrame, table1: pd.DataFrame, table3: pd.DataFrame) -> None:
    OUT_DIR.mkdir(parents=True, exist_ok=True)

    groups = [summarize_pair(df, 'overall')]
    for task in sorted(df['task'].unique()):
        groups.append(summarize_pair(df[df['task'] == task], task))
    summary = pd.DataFrame(groups)
    summary.to_csv(OUT_DIR / 'knee_summary.csv', index=False)

    ref_row = table1.iloc[0]
    enc_rmse, enc_rmse_std = _parse_mean_std(ref_row['rmse_encoder_mean_std'])
    enc_r2, enc_r2_std = _parse_mean_std(ref_row['r2_encoder_mean_std'])
    vic_rmse, vic_rmse_std = _parse_mean_std(ref_row['rmse_vicon_mean_std'])
    vic_r2, vic_r2_std = _parse_mean_std(ref_row['r2_vicon_mean_std'])

    o = summary.loc[summary.group == 'overall'].iloc[0]
    ra = summary.loc[summary.group == 'RA'].iloc[0] if 'RA' in summary.group.values else None
    rd = summary.loc[summary.group == 'RD'].iloc[0] if 'RD' in summary.group.values else None

    print(f'Reference: {REF_DIR}')
    print('\n=== table1_summary (reference) ===')
    print(table1.to_string(index=False))
    print('\n=== table3_overall (reference) ===')
    print(table3.to_string(index=False))
    print('\n=== Knee: encoder replay vs Vicon IK (from table2) ===')
    print(summary.to_string(index=False))

    rmse_pct = (enc_rmse - vic_rmse) / enc_rmse * 100.0
    r2_delta = vic_r2 - enc_r2
    latex = f"""% Knee exoskeleton — kinematic input quality (from analysis/paper_outputs/knee_exo)
Encoder replay (deployed sensor inputs offline): RMSE {enc_rmse:.3f} ± {enc_rmse_std:.3f}~Nm/kg, $R^2$ {enc_r2:.3f} ± {enc_r2_std:.3f}.
Vicon IK oracle (same TCN checkpoint): RMSE {vic_rmse:.3f} ± {vic_rmse_std:.3f}~Nm/kg, $R^2$ {vic_r2:.3f} ± {vic_r2_std:.3f}
({rmse_pct:.1f}\\% RMSE reduction, absolute $R^2$ increase {r2_delta:.3f}).
"""
    if ra is not None and rd is not None:
        latex += (
            f"Ramp ascent: encoder RMSE {ra['rmse_sensor_nmpkg']:.3f}~Nm/kg, $R^2$ {ra['r2_sensor']:.3f};"
            f" Vicon RMSE {ra['rmse_vicon_nmpkg']:.3f}~Nm/kg, $R^2$ {ra['r2_vicon']:.3f}.\n"
            f"Ramp descent: encoder RMSE {rd['rmse_sensor_nmpkg']:.3f}~Nm/kg, $R^2$ {rd['r2_sensor']:.3f};"
            f" Vicon RMSE {rd['rmse_vicon_nmpkg']:.3f}~Nm/kg, $R^2$ {rd['r2_vicon']:.3f}.\n"
        )
    (OUT_DIR / 'knee_paper_snippet.tex').write_text(latex)
    print('\nWrote', OUT_DIR / 'knee_paper_snippet.tex')
    print(latex)


table1, table2, table3 = load_reference_tables()
replay = pd.read_csv(REPLAY_METRICS_CSV)
per_trial = attach_trial_stems(table2, replay)
df = build_metrics_df(per_trial)
df.to_csv(METRICS_CSV, index=False)
write_paper_outputs(df, table1, table3)
print(f'\nWrote {METRICS_CSV} ({len(df)} trials)')
df[['trial', 'subject', 'task', 'rmse_sensor_nmpkg', 'rmse_vicon_nmpkg', 'r2_sensor_nmpkg', 'r2_vicon_nmpkg']].head()


Reference: /home/metamobility3/Jinwoo/os_kinetics/analysis/paper_outputs/knee_exo

=== table1_summary (reference) ===
 joint  n_trials_vicon  n_trials_encoder rmse_encoder_mean_std r2_encoder_mean_std rmse_vicon_mean_std r2_vicon_mean_std
Knee R              16                16         0.192 ± 0.039       0.615 ± 0.141       0.153 ± 0.036     0.740 ± 0.158

=== table3_overall (reference) ===
  replay_method        unit rmse_mean_std   r2_mean_std  n_trials
 Encoder replay N·m/kg / R² 0.192 ± 0.039 0.615 ± 0.141        16
Vicon IK oracle N·m/kg / R² 0.153 ± 0.036 0.740 ± 0.158        16

=== Knee: encoder replay vs Vicon IK (from table2) ===
  group  n_trials  rmse_sensor_nmpkg  r2_sensor  rmse_vicon_nmpkg  r2_vicon  rmse_reduction_pct  r2_increase
overall        16           0.192350   0.615256          0.153419  0.739831           20.239797     0.124575
     RA         8           0.168088   0.650550          0.135238  0.765225           19.543393     0.114675
     RD         8      

,trial,subject,task,rmse_sensor_nmpkg,rmse_vicon_nmpkg,r2_sensor_nmpkg,r2_vicon_nmpkg
0,ab01_jinwoo_knee_0p8mps_ra_exo_on,AB01_Jinwoo,RA,0.1344,0.1133,0.8133,0.8673
1,ab01_jinwoo_knee_0p8mps_rd_exo_on,AB01_Jinwoo,RD,0.2287,0.1534,0.6491,0.8420
2,ab02_oscar_knee_0p8mps_ra_exo_on,AB02_Oscar,RA,0.2107,0.1095,0.5607,0.8814
3,ab02_oscar_knee_0p8mps_rd_exo_on,AB02_Oscar,RD,0.2309,0.1657,0.6399,0.8146
4,ab03_ilseung_knee_0p8mps_ra_exo_on,AB03_Ilseung,RA,0.1778,0.1712,0.7282,0.7481


## Notes

- Metrics are **not recomputed** here; they come from `analysis/paper_outputs/knee_exo/` (same pipeline as `compare_processed_knee_exo_id.ipynb` Section 3).
- **Encoder replay** = offline sensor-path inputs (logged encoder angle + raw velocity, no input LPF).
- **Vicon oracle** = Vicon IK angle + B-spline velocity through the same TCN vs the same GT.
- To refresh numbers, re-run `compare_processed_knee_exo_id.ipynb` then this notebook.
